In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
# ライブラリ＆データ読み込み
!pip install -q keras-nlp --upgrade
!pip install japanize-matplotlib
import os, re, string, matplotlib.pyplot as plt,japanize_matplotlib, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report, confusion_matrix

train = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
test  = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")



In [ ]:
# データの基本情報
print("学習データのサイズ:", train.shape)
print("カラム一覧:", train.columns.tolist())

# 欠損値の確認
print("欠損値の数：")
print(train.isnull().sum())

# target（目的変数）のクラス分布
target_dist = train["target"].value_counts(normalize=True)
print("targetのクラス分布（割合）:")
print(target_dist)

In [ ]:
# データ前処理
train = train.fillna("")
test = test.fillna("")

import re

def clean_text(text):
    text = text.lower()  # 小文字化
    text = re.sub(r"http\S+|www\S+", " [URL] ", text)  # URLを統一表現に
    text = re.sub(r"@\w+", " [USER] ", text)           # メンションを統一
    text = re.sub(r"#", "", text)                      # ハッシュ記号だけ除去（トークンは残す）
    text = re.sub(r"[^a-z0-9\s\[\]]", "", text)         # 英数字と[URL]/[USER]以外を除去
    text = re.sub(r"\s+", " ", text).strip()           # 余計なスペース除去
    return text

# text列に前処理を適用
train["text"] = train["text"].apply(clean_text)
test["text"] = test["text"].apply(clean_text)


**# 仮説：# 災害関連のツイートには、文脈的な特徴（語順・文法・感情）が現れる。# 単語の出現頻度だけでなく、文の意味を捉えるモデルが有効と考える。# 説明変数：# 文全体のテキストを DistilBERT に入力し、文脈情報を抽出して分類に用いる。**


In [ ]:
# DistilBERT fine-tuning
import tensorflow as tf
import keras_nlp
from keras_nlp.models import DistilBertClassifier
from sklearn.model_selection import train_test_split

# データの準備
X = train["text"]
y = train["target"]

# データ分割（80% train, 20% val）
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# DataFrame形式に整形（textとtarget列を持たせる）
train_df = pd.DataFrame({"text": X_train, "target": y_train})
val_df   = pd.DataFrame({"text": X_val, "target": y_val})

# モデル設定
PRESET = "distil_bert_base_en_uncased"
BATCH  = 32
EPOCHS = 2

# KerasNLP の前処理レイヤとエンコーダー
preproc = keras_nlp.models.DistilBertPreprocessor.from_preset(PRESET)
encoder = keras_nlp.models.DistilBertBackbone.from_preset(PRESET)

# 分類器本体
classifier = DistilBertClassifier(
    backbone=encoder,
    num_classes=2,
    activation="softmax",
)

# データセット作成関数
def make_ds(df, label=True):
    texts = df["text"].values.tolist()
    if label:
        labels = df["target"].values
        return tf.data.Dataset.from_tensor_slices((texts, labels)).batch(BATCH).map(
            lambda x, y: (preproc(x), y)
        )
    else:
        return tf.data.Dataset.from_tensor_slices(texts).batch(BATCH).map(preproc)

# データセット作成
train_ds = make_ds(train_df)
val_ds   = make_ds(val_df)

# モデルのコンパイルと学習
classifier.compile(
    optimizer=tf.keras.optimizers.Adam(5e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

classifier.fit(train_ds, epochs=EPOCHS, validation_data=val_ds)


In [ ]:
# 予測：val_df を使う
val_pred_bert = np.argmax(classifier.predict(make_ds(val_df)), axis=1)

# ヒートマップで混同行列を表示
sns.heatmap(confusion_matrix(val_df["target"], val_pred_bert), annot=True, fmt="d", cmap="Blues")
plt.title("DistilBERT Confusion Matrix")
plt.xlabel("Pred")
plt.ylabel("True")
plt.show()

In [ ]:
# 詳細な分類レポートを出力
print(classification_report(val_df["target"], val_pred_bert))

In [ ]:
# テストデータに対する検証
test_pred  = np.argmax(classifier.predict(make_ds(test, label=False)), axis=1)

sub = pd.read_csv("/kaggle/input/nlp-getting-started/sample_submission.csv")
sub["target"] = test_pred
sub.to_csv("submission.csv", index=False)
sub.head()
